# Radar Interleaver Demo with RF Imperfections
This notebook demonstrates the synthetic Pulse Descriptor Word (PDW) generation with **interleaving**, **missing pulses (POI & dead-time)**, and **false pulses (noise & multipath)**.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from radar_generator import Environment, StableRadar, JitterRadar, StaggerRadar, SlidingRadar

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Setting up the Environment with Imperfections

In [ ]:
env = Environment()

# 1. Stable PRI with low POI (missing a lot of pulses) and multipath echoes
stable_radar = StableRadar(
    radar_id="R_STABLE", 
    pri=1000.0, 
    freq_mean=9000.0, 
    freq_mode="hopping", 
    freq_hopping_values=[9000.0, 9100.0, 9200.0],
    poi=0.70,               # Only 70% of pulses arrive at the receiver
    multipath_prob=0.10,    # 10% chance a pulse generates a reflection
    multipath_delay_mean=1.5
)
env.add_radar(stable_radar)

# 2. Jitter PRI with high POI
jitter_radar = JitterRadar(
    radar_id="R_JITTER", 
    pri_mean=1500.0, 
    pri_jitter_std=50.0, 
    freq_mean=9500.0,
    poi=0.95
)
env.add_radar(jitter_radar)

# 3. Stagger PRI with perfect POI (strong signal)
stagger_radar = StaggerRadar(
    radar_id="R_STAGGER", 
    stagger_pris=[800.0, 1200.0, 1500.0], 
    freq_mean=8500.0, 
    freq_mode="agility", 
    freq_hopping_values=[8500.0, 8600.0, 8700.0, 8800.0],
    poi=1.0
)
env.add_radar(stagger_radar)

# Generate 50,000 microseconds (50 ms) of interleaved data
# Add global receiver dead-time (drop 5% of all pulses) and inject 1000 noise pulses per second
df_interleaved = env.generate(
    duration=50000.0, 
    toa_noise_std=5.0, 
    freq_noise_std=2.0,
    global_drop_rate=0.05,
    noise_rate=1000.0
)

print(f"Total pulses generated: {len(df_interleaved)}")
display(df_interleaved.head(15))

## 2. Visualization

In [ ]:
plt.figure(figsize=(12, 6))

# Use seaborn for easy color coding by Radar_ID
sns.scatterplot(data=df_interleaved, x='TOA', y='Frequency', hue='Radar_ID', palette='Set1', s=60, marker='x')

plt.title("Interleaved Radar Environment with Missing/False Pulses (TOA vs Frequency)")
plt.xlabel("Time of Arrival (us)")
plt.ylabel("Frequency (MHz)")
plt.grid(True)
plt.legend(title='Radar Emitter', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

# Plot the Raw Measured Delta-TOA 
# Notice how the missing pulses create large spikes in Measured_Delta_TOA (2x or 3x the normal PRI)
plt.plot(df_interleaved['TOA'], df_interleaved['Measured_Delta_TOA'], color='gray', linestyle='-', alpha=0.5, label="Measured Delta TOA")
sns.scatterplot(data=df_interleaved, x='TOA', y='Measured_Delta_TOA', hue='Radar_ID', palette='Set1', s=50)

plt.title("Interleaved Radar Environment (TOA vs Measured Delta TOA)")
plt.xlabel("Time of Arrival (us)")
plt.ylabel("Delta TOA from previous pulse (us)")
plt.grid(True)
plt.legend(title='Ground Truth', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()